# ContractRisk M3 — Auditoría RAGAS 1.1
Activa GPU T4. Sube `contractrisk_m3_auditoria_codigo.zip`. Esta corrida reevalúa respuestas originales; no genera otras ni modifica gold. La calidad semántica del juez aún requiere revisión humana.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, tempfile, os, sys, subprocess, shutil
uploaded = files.upload()
assert len(uploaded) == 1, 'Sube solo el ZIP de código de auditoría'
archive = next(iter(uploaded))
ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_m3_audit_', dir='/content'))
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        target = (ROOT / member.filename).resolve()
        assert target.is_relative_to(ROOT), 'Ruta no permitida en ZIP'
    z.extractall(ROOT)
os.chdir(ROOT)
assert (ROOT/'m3/run_ragas_audit_m3.py').is_file()
print(ROOT)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'm3/requirements-development-v2.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Activa GPU T4'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], check=True)


In [ ]:
OUTPUT = ROOT/'ragas_auditoria_resultados'
assert not OUTPUT.exists(), 'No sobrescribir una corrida previa; reinicia desde la carga para repetir'
with (ROOT/'auditoria_execution.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen([sys.executable, '-u', 'm3/run_ragas_audit_m3.py', '--output', str(OUTPUT)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end=''); log.write(line); log.flush()
    returncode = process.wait()
print('Código de salida:', returncode)
print('Ejecuta la siguiente celda también si hubo error.')


In [ ]:
import json
if (OUTPUT/'metadata.json').exists():
    meta = json.loads((OUTPUT/'metadata.json').read_text(encoding='utf-8'))
    print(json.dumps(meta.get('summaries', {'status': meta['status'], 'error': meta.get('error')}), indent=2, ensure_ascii=False))
OUTPUT.mkdir(exist_ok=True)
shutil.copy2(ROOT/'auditoria_execution.log', OUTPUT/'execution.log')
result_zip = shutil.make_archive(str(ROOT/'contractrisk_m3_auditoria_resultados'), 'zip', OUTPUT)
files.download(result_zip)
